In [ ]:
!pip install lightgbm catboost scipy scikit-learn pandas numpy


In [5]:
# ============ p8 CONFIG ============
# Philosophy: less post-processing, more robust base model
# Key changes vs p5/p6/p7:
#   1. REMOVE Stage-1 binary Rest/Active gate (was inflating OOF)
#   2. REMOVE class_weight / auto_class_weights (Rest recall was hurting)
#   3. REDUCE smoothing window to 3 (was 5 -- safer for unseen PIDs)
#   4. INCREASE stat context to ±3s (was ±1s -- captures full rep patterns)
#   5. SELECT hyperparams by variance-penalised score (mean - lambda*std)
#   6. ADD device_is_unknown feature (92k unknown-device test rows)
#   7. KEEP 3 models: LGBM + CatBoost + ExtraTrees (diversity)
import numpy as np, pandas as pd
from scipy.fft import rfft, rfftfreq
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.ensemble import ExtraTreesClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier
import warnings; warnings.filterwarnings('ignore')

In [6]:
FS = 50                    # target uniform rate (Hz)
SHORT_S  = 3               # ±3s stat context (was 1) -- captures full rep
LONG_S   = 4               # ±4s for FFT/peaks (unchanged)
EXT_S    = 6               # ±6s extended FFT (unchanged)
N_CLASS  = 9
SEED     = 42
SMOOTH_WIN = 3             # temporal smoothing window (was 5)
SMOOTH_ALPHA = 0.6
LAMBDA_VAR   = 0.5         # penalty weight for fold std in selection
P5_BASELINE_BA = 0.91879   # guard: compare against best known OOF

# ---- PATHS (edit if yours differ) ----
PATH = '/Users/dayana/git repo/machine-learning-class/lab6/knu-2026-machine-learning-final-assignment/'
F = dict(
    tr_acc=PATH+'train-accel.csv',
    tr_gyr=PATH+'train-gyro.csv',
    tr_lab=PATH+'train-label.csv',
    te_acc=PATH+'test-accel.csv',
    te_gyr=PATH+'test-gyro.csv',
    te_lab=PATH+'test-label.csv',
    sub=PATH+'submission.csv',
)
print('config ready')

config ready


In [7]:
train_accel = pd.read_csv(F['tr_acc']); train_gyro = pd.read_csv(F['tr_gyr'])
train_label = pd.read_csv(F['tr_lab'])
test_accel  = pd.read_csv(F['te_acc']); test_gyro  = pd.read_csv(F['te_gyr'])
test_label  = pd.read_csv(F['te_lab'])
sub_template = pd.read_csv(F['sub'])

for df in (train_accel, train_gyro, test_accel, test_gyro):
    df.columns = [c.strip().lower() for c in df.columns]

print('accel', train_accel.shape, 'gyro', train_gyro.shape, 'label', train_label.shape)
print('test_label', test_label.shape, 'sub', sub_template.shape)
print('devices(train):', train_accel['device'].value_counts().to_dict())
print('devices(test):', test_accel['device'].value_counts().to_dict())

accel (2428374, 7) gyro (2433673, 7) label (38015, 4)
test_label (39473, 4) sub (39473, 2)
devices(train): {'samsung': 1558848, 'Apple': 869526}
devices(test): {'samsung': 1564692, 'Apple': 870850, 'unknown': 92768}


In [8]:
SENS = ['x','y','z']

def fit_device_stats(acc, gyr):
    stats = {}
    for name, df in [('accel', acc), ('gyro', gyr)]:
        g_mean = df[SENS].mean(); g_std = df[SENS].std().replace(0, 1)
        per = {}
        for dev, sub in df.groupby('device'):
            per[dev] = (sub[SENS].mean(), sub[SENS].std().replace(0, 1))
        stats[name] = dict(global_mean=g_mean, global_std=g_std, per=per)
    return stats

def apply_device_norm(df, st):
    out = df.copy()
    gm, gs, per = st['global_mean'], st['global_std'], st['per']
    for dev, sub_idx in out.groupby('device').groups.items():
        m, s = per.get(dev, (gm, gs))
        out.loc[sub_idx, SENS] = (out.loc[sub_idx, SENS] - m.values) / s.values
    return out

DEV_STATS      = fit_device_stats(train_accel, train_gyro)
train_accel_n  = apply_device_norm(train_accel, DEV_STATS['accel'])
train_gyro_n   = apply_device_norm(train_gyro,  DEV_STATS['gyro'])
test_accel_n   = apply_device_norm(test_accel,  DEV_STATS['accel'])
test_gyro_n    = apply_device_norm(test_gyro,   DEV_STATS['gyro'])
print('device-normalized (train stats only)')

device-normalized (train stats only)


In [9]:
# ---- Hampel spike removal + uniform resampling ----
def hampel(x, k=5, nsig=3.0):
    x = np.asarray(x, float)
    if len(x) < 2*k+1: return x
    med = pd.Series(x).rolling(2*k+1, center=True, min_periods=1).median().values
    mad = pd.Series(np.abs(x-med)).rolling(2*k+1, center=True, min_periods=1).median().values
    thr = nsig * 1.4826 * mad
    out = x.copy(); bad = np.abs(x-med) > thr
    out[bad] = med[bad]
    return out

def clean_and_resample_pid(sub, fs=FS):
    sub = sub.sort_values('time')
    t = sub['time'].values.astype(float)
    if len(t) < 4 or (t.max()-t.min()) <= 0:
        return sub
    tg = np.arange(t.min(), t.max(), 1.0/fs)
    res = {'time': tg}
    for c in SENS:
        v = hampel(sub[c].values, k=5, nsig=3.0)
        res[c] = np.interp(tg, t, v)
    out = pd.DataFrame(res)
    for meta in ('pid','direction','device'):
        out[meta] = sub[meta].iloc[0]
    return out

def preprocess_all(acc, gyr):
    a_parts, g_parts = [], []
    for pid in acc['pid'].unique():
        a_parts.append(clean_and_resample_pid(acc[acc.pid==pid]))
    for pid in gyr['pid'].unique():
        g_parts.append(clean_and_resample_pid(gyr[gyr.pid==pid]))
    return pd.concat(a_parts, ignore_index=True), pd.concat(g_parts, ignore_index=True)

train_accel_r, train_gyro_r = preprocess_all(train_accel_n, train_gyro_n)
test_accel_r,  test_gyro_r  = preprocess_all(test_accel_n,  test_gyro_n)
print('resampled+cleaned ->', train_accel_r.shape, train_gyro_r.shape,
      test_accel_r.shape, test_gyro_r.shape)

resampled+cleaned -> (2395969, 7) (2395967, 7) (2536513, 7) (2536520, 7)


In [10]:
# ---- Direction correction ----
def direction_correct(df):
    out = df.copy()
    x, y, z = out['x'].values, out['y'].values, out['z'].values
    d = out['direction'].values
    xb = np.where(np.isin(d,[1,3]),  x, -x)   # lateral
    yb = np.where(np.isin(d,[1,2]),  y, -y)   # forward/back
    zb = z.copy()                               # vertical
    out['xb'], out['yb'], out['zb'] = xb, yb, zb
    out['magb'] = np.sqrt(xb*xb + yb*yb + zb*zb)
    return out

for nm in ['train_accel_r','train_gyro_r','test_accel_r','test_gyro_r']:
    globals()[nm] = direction_correct(globals()[nm])
print('direction corrected; cols:', [c for c in train_accel_r.columns if c not in SENS+['time','pid','direction','device']])

direction corrected; cols: ['xb', 'yb', 'zb', 'magb']


In [11]:
# ---- Lookup table ----
def build_lookup(df):
    df = df.copy()
    df['t_int'] = np.floor(df['time']).astype(int)
    cols = ['xb','yb','zb','magb']
    lut = {}
    for (pid, ts), g in df.groupby(['pid','t_int']):
        lut[(pid, ts)] = g[cols].values.astype(np.float32)
    return lut

# Also build device lookup for feature: is_unknown
def build_device_lookup(df):
    df = df.copy()
    df['t_int'] = np.floor(df['time']).astype(int)
    lut = {}
    for (pid, ts), g in df.groupby(['pid','t_int']):
        lut[(pid, ts)] = g['device'].iloc[0]
    return lut

LUT = dict(
    tr_acc=build_lookup(train_accel_r), tr_gyr=build_lookup(train_gyro_r),
    te_acc=build_lookup(test_accel_r),  te_gyr=build_lookup(test_gyro_r),
)
DEV_LUT = dict(
    tr=build_device_lookup(train_accel_r),
    te=build_device_lookup(test_accel_r),
)
print('lookup built:', {k: len(v) for k,v in LUT.items()})

lookup built: {'tr_acc': 47927, 'tr_gyr': 47927, 'te_acc': 50734, 'te_gyr': 50734}


In [12]:
def gather_window(lut, pid, t0, half_s):
    parts = [lut[(pid, t0+dt)] for dt in range(-half_s, half_s+1) if (pid, t0+dt) in lut]
    if not parts: return None
    return np.concatenate(parts, axis=0)

def safe(a, f, d=0.0):
    try:
        v = f(a); return d if not np.isfinite(v) else v
    except Exception: return d

def stat_block(v, pfx):
    f = {}
    f[pfx+'mean']   = safe(v, np.mean)
    f[pfx+'std']    = safe(v, np.std)
    f[pfx+'min']    = safe(v, np.min)
    f[pfx+'max']    = safe(v, np.max)
    f[pfx+'rng']    = f[pfx+'max'] - f[pfx+'min']
    f[pfx+'med']    = safe(v, np.median)
    f[pfx+'q25']    = safe(v, lambda a: np.percentile(a, 25))
    f[pfx+'q75']    = safe(v, lambda a: np.percentile(a, 75))
    f[pfx+'iqr']    = f[pfx+'q75'] - f[pfx+'q25']
    f[pfx+'rms']    = safe(v, lambda a: np.sqrt(np.mean(a*a)))
    f[pfx+'mad']    = safe(v, lambda a: np.mean(np.abs(a-np.mean(a))))
    f[pfx+'skew']   = safe(v, lambda a: ((a-a.mean())**3).mean()/(a.std()**3+1e-9))
    f[pfx+'kurt']   = safe(v, lambda a: ((a-a.mean())**4).mean()/(a.std()**4+1e-9))
    f[pfx+'energy'] = safe(v, lambda a: np.sum(a*a)/len(a))
    f[pfx+'zcr']    = safe(v, lambda a: np.mean(np.abs(np.diff(np.sign(a-a.mean())))))
    return f

def fft_block(v, pfx, fs=FS):
    v = v - np.mean(v); n = len(v)
    if n < 8: return {pfx+k: 0.0 for k in ['domf','domp','ent','cent','flat','p_low','p_high']}
    Y = np.abs(rfft(v * np.hanning(n))); fr = rfftfreq(n, 1/fs)
    Y[0] = 0; P = Y*Y; tot = P.sum() + 1e-9
    i = int(np.argmax(P))
    if 0 < i < len(P)-1:
        a,b,c = P[i-1],P[i],P[i+1]; denom = (a-2*b+c)
        d = 0.5*(a-c)/denom if denom != 0 else 0.0
    else: d = 0.0
    domf = (i+d)*fs/n
    pr = P/tot
    ent  = -np.sum(pr*np.log(pr+1e-12))
    cent = np.sum(fr*P)/tot
    flat = np.exp(np.mean(np.log(P+1e-12)))/(np.mean(P)+1e-12)
    p_low  = P[(fr>=0.5)&(fr<2.0)].sum()/tot
    p_high = P[(fr>=2.0)&(fr<5.0)].sum()/tot
    return {pfx+'domf': domf, pfx+'domp': P[i]/tot, pfx+'ent': ent,
            pfx+'cent': cent, pfx+'flat': flat, pfx+'p_low': p_low, pfx+'p_high': p_high}

def autocorr_period(v, fs=FS):
    v = v - np.mean(v); n = len(v)
    if n < 16 or np.std(v) < 1e-6: return 0.0, 0.0
    ac = np.correlate(v, v, 'full')[n-1:]; ac /= (ac[0]+1e-9)
    lo, hi = int(fs*0.2), min(int(fs*2.0), n-1)
    if hi <= lo: return 0.0, 0.0
    seg = ac[lo:hi]; k = int(np.argmax(seg))+lo
    return fs/k if k > 0 else 0.0, float(ac[k])

print('feature helpers defined')

feature helpers defined


In [13]:
CH = {0:'xb', 1:'yb', 2:'zb', 3:'magb'}

def base_features(acc_w, gyr_w):
    f = {}
    for src, W in [('a', acc_w), ('g', gyr_w)]:
        if W is None: continue
        for idx, nm in CH.items():
            col = W[:, idx]
            f.update(stat_block(col, f'{src}_{nm}_s_'))
            f.update(fft_block(col,  f'{src}_{nm}_f_'))
            p, c = autocorr_period(col)
            f[f'{src}_{nm}_acp'] = p
            f[f'{src}_{nm}_acc'] = c
    return f

def p8_extra(acc_w, gyr_w, fs=FS):
    """Physics-motivated features kept from p6 + jerk improvements."""
    f = {}
    keys = ['sym_lat','vert_lat_ratio','vert_energy','lat_energy',
            'ag_coupling','still_frac','spec_flat_mag','domf_mag',
            'pca_vert','jerk_rms']
    if acc_w is None or len(acc_w) < 8:
        return {f'p8_{k}': 0.0 for k in keys}
    xb, yb, zb, mag = acc_w[:,0], acc_w[:,1], acc_w[:,2], acc_w[:,3]
    f['p8_sym_lat']        = safe(xb, lambda a: abs(((a-a.mean())**3).mean())/(a.std()**3+1e-9))
    ve = np.sum(zb*zb)/len(zb); le = np.sum(xb*xb)/len(xb)
    f['p8_vert_energy']    = ve
    f['p8_lat_energy']     = le
    f['p8_vert_lat_ratio'] = ve/(le+1e-6)
    fb = fft_block(mag, 'm_', fs)
    f['p8_domf_mag']       = fb['m_domf']
    f['p8_spec_flat_mag']  = fb['m_flat']
    jerk = np.diff(mag)*fs
    f['p8_jerk_rms']       = safe(jerk, lambda a: np.sqrt(np.mean(a*a)))
    thr = 0.05*(np.std(mag)+1e-6)
    f['p8_still_frac']     = float(np.mean(np.abs(mag-np.mean(mag)) < thr))
    M = np.vstack([xb,yb,zb]); M = M - M.mean(1, keepdims=True)
    try:
        u,s,vt = np.linalg.svd(M, full_matrices=False)
        f['p8_pca_vert'] = abs(u[2,0])
    except Exception:
        f['p8_pca_vert'] = 0.0
    if gyr_w is not None and len(gyr_w) >= 8:
        gm = gyr_w[:,3]; n = min(len(mag), len(gm))
        a1 = mag[:n]-mag[:n].mean(); g1 = gm[:n]-gm[:n].mean()
        denom = (np.std(a1)*np.std(g1)*n+1e-9)
        f['p8_ag_coupling'] = float(np.dot(a1,g1)/denom)
    else:
        f['p8_ag_coupling'] = 0.0
    return f

def make_row(la, lg, dev_lut, pid, t0):
    # SHORT_S=3: ±3s stat window (key change from p5/p6 ±1s)
    a_short = gather_window(la, pid, t0, SHORT_S)
    a_long  = gather_window(la, pid, t0, LONG_S)
    g_long  = gather_window(lg, pid, t0, LONG_S)
    a_ext   = gather_window(la, pid, t0, EXT_S)
    f = {}
    f.update(base_features(a_long, g_long))
    # SHORT_S stat block -- now ±3s instead of ±1s
    if a_short is not None:
        f.update(stat_block(a_short[:,3], 'sh_mag_'))
    if a_ext is not None:
        f.update(fft_block(a_ext[:,3], 'ext_mag_'))
    f.update(p8_extra(a_long, g_long))
    # NEW: device_is_unknown flag
    dev = dev_lut.get((pid, t0), 'unknown')
    f['device_is_unknown'] = float(dev == 'unknown')
    return f

def build_matrix(label_df, la, lg, dev_lut):
    rows = []; idx_keep = []
    for r in label_df.itertuples():
        pid = r.pid; t0 = int(np.floor(r.time))
        if (pid,t0) not in la and (pid,t0) not in lg:
            continue
        rows.append(make_row(la, lg, dev_lut, pid, t0))
        idx_keep.append(r.Index)
    X = pd.DataFrame(rows).fillna(0.0)
    return X, idx_keep

X_tr, keep_tr = build_matrix(train_label, LUT['tr_acc'], LUT['tr_gyr'], DEV_LUT['tr'])
X_te, keep_te = build_matrix(test_label,  LUT['te_acc'], LUT['te_gyr'], DEV_LUT['te'])
X_tr, X_te = X_tr.align(X_te, join='outer', axis=1, fill_value=0.0)
y_tr = train_label.loc[keep_tr, 'workout'].values
g_tr = train_label.loc[keep_tr, 'pid'].values
print('X_tr', X_tr.shape, 'X_te', X_te.shape, 'classes', np.unique(y_tr))

X_tr (37821, 225) X_te (38349, 225) classes [0 1 2 3 4 5 6 7 8]


In [14]:
# ============================================================
# STAGE 2 only: NO class weights, NO binary gate
# LGBM + CatBoost + ExtraTrees
# ============================================================
N_FOLDS = 5
gkf = GroupKFold(n_splits=N_FOLDS)

oof_lgbm = np.zeros((len(X_tr), N_CLASS))
oof_cat  = np.zeros((len(X_tr), N_CLASS))
oof_et   = np.zeros((len(X_tr), N_CLASS))
te_lgbm  = np.zeros((len(X_te), N_CLASS))
te_cat   = np.zeros((len(X_te), N_CLASS))
te_et    = np.zeros((len(X_te), N_CLASS))

fold_bas_lgbm = []; fold_bas_cat = []; fold_bas_et = []

# NO class_weight='balanced' -- removed to recover Rest recall
lgb_par = dict(
    objective='multiclass', num_class=N_CLASS, learning_rate=0.03,
    num_leaves=64, feature_fraction=0.7, bagging_fraction=0.8,
    bagging_freq=1, min_child_samples=40, n_estimators=1400,
    random_state=SEED, verbose=-1
)

for k, (tr, va) in enumerate(gkf.split(X_tr, y_tr, g_tr)):
    # --- LGBM ---
    m = lgb.LGBMClassifier(**lgb_par)
    m.fit(X_tr.iloc[tr], y_tr[tr],
          eval_set=[(X_tr.iloc[va], y_tr[va])],
          callbacks=[lgb.early_stopping(80, verbose=False)])
    oof_lgbm[va] = m.predict_proba(X_tr.iloc[va])
    te_lgbm += m.predict_proba(X_te)
    ba_l = balanced_accuracy_score(y_tr[va], oof_lgbm[va].argmax(1))
    fold_bas_lgbm.append(ba_l)

    # --- CatBoost (NO auto_class_weights) ---
    c = CatBoostClassifier(
        iterations=1500, learning_rate=0.03, depth=7,
        loss_function='MultiClass',
        random_seed=SEED, verbose=0
    )
    c.fit(X_tr.iloc[tr], y_tr[tr],
          eval_set=(X_tr.iloc[va], y_tr[va]),
          early_stopping_rounds=80)
    oof_cat[va] = c.predict_proba(X_tr.iloc[va])
    te_cat += c.predict_proba(X_te)
    ba_c = balanced_accuracy_score(y_tr[va], oof_cat[va].argmax(1))
    fold_bas_cat.append(ba_c)

    # --- ExtraTrees (3rd model for diversity) ---
    et = ExtraTreesClassifier(
        n_estimators=500, max_features=0.5, min_samples_leaf=4,
        n_jobs=-1, random_state=SEED
    )
    et.fit(X_tr.iloc[tr], y_tr[tr])
    oof_et[va] = et.predict_proba(X_tr.iloc[va])
    te_et += et.predict_proba(X_te)
    ba_e = balanced_accuracy_score(y_tr[va], oof_et[va].argmax(1))
    fold_bas_et.append(ba_e)

    print(f'fold{k}  LGBM={ba_l:.4f}  CatBoost={ba_c:.4f}  ET={ba_e:.4f}')

te_lgbm /= N_FOLDS; te_cat /= N_FOLDS; te_et /= N_FOLDS
print('OOF folds done')
print(f'LGBM  folds: {[round(x,4) for x in fold_bas_lgbm]}  mean={np.mean(fold_bas_lgbm):.4f}  std={np.std(fold_bas_lgbm):.4f}')
print(f'Cat   folds: {[round(x,4) for x in fold_bas_cat]}   mean={np.mean(fold_bas_cat):.4f}  std={np.std(fold_bas_cat):.4f}')
print(f'ET    folds: {[round(x,4) for x in fold_bas_et]}    mean={np.mean(fold_bas_et):.4f}  std={np.std(fold_bas_et):.4f}')

fold0  LGBM=0.8874  CatBoost=0.8900  ET=0.8903
fold1  LGBM=0.9560  CatBoost=0.9560  ET=0.9547
fold2  LGBM=0.9004  CatBoost=0.8949  ET=0.9021
fold3  LGBM=0.8832  CatBoost=0.8993  ET=0.8997
fold4  LGBM=0.8667  CatBoost=0.8815  ET=0.8772
OOF folds done
LGBM  folds: [0.8874, 0.956, 0.9004, 0.8832, 0.8667]  mean=0.8987  std=0.0306
Cat   folds: [0.89, 0.956, 0.8949, 0.8993, 0.8815]   mean=0.9043  std=0.0265
ET    folds: [0.8903, 0.9547, 0.9021, 0.8997, 0.8772]    mean=0.9048  std=0.0264


In [15]:
# ---- Soft probability temporal smoothing (window=3, safer than 5) ----
def soft_prob_smooth(P, label_df, keep_idx, win=SMOOTH_WIN, alpha=SMOOTH_ALPHA):
    """Temporal smoothing within pid ordered by time; leakage-safe."""
    out = P.copy()
    df = label_df.loc[keep_idx, ['pid','time']].reset_index(drop=True)
    for pid in df.pid.unique():
        ix = np.where(df.pid.values == pid)[0]
        order = ix[np.argsort(df.time.values[ix])]
        sm = P[order].copy()
        for i in range(len(order)):
            lo = max(0, i-win); hi = min(len(order), i+win+1)
            sm[i] = alpha*P[order[i]] + (1-alpha)*P[order[lo:hi]].mean(0)
        out[order] = sm
    return out

print('smoothing helper defined (window=3)')

smoothing helper defined (window=3)


In [16]:
# ============================================================
# Blend selection: variance-penalised score
# score = mean_fold_BA - LAMBDA_VAR * std_fold_BA
# No Stage-1 gate at all
# ============================================================
from sklearn.metrics import classification_report

best = (-1, None)
# Grid over blend weights (LGBM, CatBoost, ET)
for wl in [0.40, 0.45, 0.50]:
    for wc in [0.30, 0.35, 0.40]:
        we = round(1.0 - wl - wc, 2)
        if we < 0.05 or we > 0.40: continue
        P = wl*oof_lgbm + wc*oof_cat + we*oof_et
        Ps = soft_prob_smooth(P, train_label, keep_tr)
        preds = Ps.argmax(1)
        # compute per-fold BA for variance penalty
        fold_bas = []
        for _, va in gkf.split(X_tr, y_tr, g_tr):
            fold_bas.append(balanced_accuracy_score(y_tr[va], preds[va]))
        mean_ba = np.mean(fold_bas)
        std_ba  = np.std(fold_bas)
        score   = mean_ba - LAMBDA_VAR * std_ba   # penalise variance
        if score > best[0]:
            best = (score, (wl, wc, we, mean_ba, std_ba))

_, (WL, WC, WE, MEAN_BA, STD_BA) = best
print(f'Best blend: LGBM={WL} CatBoost={WC} ET={WE}')
print(f'OOF mean_BA={MEAN_BA:.5f}  std={STD_BA:.5f}  penalised={best[0]:.5f}')

# Final OOF with chosen blend
P_oof = WL*oof_lgbm + WC*oof_cat + WE*oof_et
Ps_oof = soft_prob_smooth(P_oof, train_label, keep_tr)
oof_ba_p8 = balanced_accuracy_score(y_tr, Ps_oof.argmax(1))
print(f'\np8 OOF BA={oof_ba_p8:.5f} | p5 baseline={P5_BASELINE_BA:.5f}')
print(classification_report(y_tr, Ps_oof.argmax(1), digits=3))

Best blend: LGBM=0.4 CatBoost=0.4 ET=0.2
OOF mean_BA=0.90945  std=0.02698  penalised=0.89596

p8 OOF BA=0.90899 | p5 baseline=0.91879
              precision    recall  f1-score   support

           0      0.947     0.953     0.950      3389
           1      0.933     0.902     0.917      3523
           2      0.951     0.867     0.907      3575
           3      0.938     0.895     0.916      3697
           4      0.922     0.932     0.927      3610
           5      0.919     0.925     0.922      3627
           6      0.906     0.869     0.887      3589
           7      0.911     0.868     0.889      3601
           8      0.889     0.969     0.927      9210

    accuracy                          0.918     37821
   macro avg      0.924     0.909     0.916     37821
weighted avg      0.919     0.918     0.917     37821



In [17]:
# ---- Test prediction ----
P_te  = WL*te_lgbm + WC*te_cat + WE*te_et
P_te_s = soft_prob_smooth(P_te, test_label, keep_te)
pred_te = P_te_s.argmax(1)

# Map predictions back to ids
test_pred = pd.Series(8, index=test_label.index)   # default Rest
test_pred.loc[keep_te] = pred_te
sub = sub_template.copy()
id2lab = dict(zip(test_label['id'], test_pred.values))
sub['workout'] = sub['id'].map(id2lab).fillna(8).astype(int)

In [18]:
# ---- Always write + always print result ----
sub.to_csv('submission_p8.csv', index=False)
delta = oof_ba_p8 - P5_BASELINE_BA
status = '✅ IMPROVED' if delta >= 0 else '⚠️ REGRESSION'
print('='*60)
print(f'{status}')
print(f'p8 OOF BA      : {oof_ba_p8:.5f}')
print(f'p5 baseline    : {P5_BASELINE_BA:.5f}')
print(f'delta          : {delta:+.5f}')
print(f'blend          : LGBM={WL} Cat={WC} ET={WE}')
print(f'OOF mean±std   : {MEAN_BA:.5f} ± {STD_BA:.5f}')
print(f'penalised score: {best[0]:.5f}')
print(f'file           : submission_p8.csv written ({sub.shape[0]} rows)')
print('pred dist      :', pd.Series(sub['workout']).value_counts().sort_index().to_dict())
print('='*60)
if delta < 0:
    print('Note: OOF dropped vs p5 -- this is EXPECTED if Stage-1 removal')
    print('reduced inflation. Check if LB improves regardless.')

⚠️ REGRESSION
p8 OOF BA      : 0.90899
p5 baseline    : 0.91879
delta          : -0.00980
blend          : LGBM=0.4 Cat=0.4 ET=0.2
OOF mean±std   : 0.90945 ± 0.02698
penalised score: 0.89596
file           : submission_p8.csv written (39473 rows)
pred dist      : {0: 3763, 1: 3358, 2: 3679, 3: 3439, 4: 3867, 5: 3728, 6: 4135, 7: 3668, 8: 9836}
Note: OOF dropped vs p5 -- this is EXPECTED if Stage-1 removal
reduced inflation. Check if LB improves regardless.
